# 🤖 UI-TARS on Lightning.ai

Deploys **UI-TARS-7B-DPO** via vLLM with an OpenAI-compatible endpoint exposed through Lightning.ai's built-in Port Viewer — no Cloudflare, no ngrok.

| Step | What it does |
|------|--------------|
| 1 | Install deps (once — cached in `~/storage`) |
| 2 | Download model (once — persists across restarts) |
| 3 | Start vLLM server on port 8000 |
| 4 | Expose via Lightning Port Viewer → get public URL |

> **GPU:** L4 (24 GB VRAM) recommended — fits 7B-DPO at full fp16, no quantization.

---
## Step 1 — Install Dependencies
_Installs to `~/storage/venv` so it survives Studio restarts._

In [ ]:
import subprocess, sys, os

VENV = os.path.expanduser("~/storage/venv")
PIP  = f"{VENV}/bin/pip"

if not os.path.exists(VENV):
    print("Creating persistent venv in ~/storage/venv ...")
    subprocess.run([sys.executable, "-m", "venv", VENV], check=True)
else:
    print("✅ venv already exists — skipping creation")

packages = [
    "--upgrade pip",
    "transformers>=4.40.0",
    "huggingface_hub",
    "accelerate",
    "vllm==0.6.6 --extra-index-url https://download.pytorch.org/whl/cu124",
]

for pkg in packages:
    print(f"\n📦 Installing: {pkg.split()[0]} ...")
    result = subprocess.run(
        f"{PIP} install {pkg}",
        shell=True, capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f"❌ FAILED:\n{result.stderr[-800:]}")
    else:
        print(f"   ✅ Done")

print("\n🎉 All dependencies installed.")

---
## Step 2 — Download Model
_Downloads to `~/storage/models/UI-TARS-7B-DPO`. Skip if already present._

In [ ]:
import os, subprocess

MODEL_ID   = "bytedance-research/UI-TARS-7B-DPO"
MODEL_DIR  = os.path.expanduser("~/storage/models/UI-TARS-7B-DPO")
HF_CLI     = os.path.expanduser("~/storage/venv/bin/huggingface-cli")

os.makedirs(MODEL_DIR, exist_ok=True)

# Check if model is already downloaded
config_path = os.path.join(MODEL_DIR, "config.json")
if os.path.exists(config_path):
    print(f"✅ Model already present at: {MODEL_DIR}")
    print("   Skipping download.")
else:
    print(f"📥 Downloading {MODEL_ID} ...")
    print(f"   Destination: {MODEL_DIR}")
    print("   This may take 10–15 min on first run.\n")

    result = subprocess.run(
        [
            HF_CLI, "download", MODEL_ID,
            "--local-dir", MODEL_DIR,
            "--local-dir-use-symlinks", "False",
        ],
        capture_output=False   # stream output live
    )

    if result.returncode == 0:
        print(f"\n✅ Download complete: {MODEL_DIR}")
    else:
        print("\n❌ Download failed. Check HF_TOKEN if model is gated.")

---
## Step 3 — Start vLLM Server
_Launches on port `8000` in the background. Logs to `~/storage/vllm.log`._

In [ ]:
import os, subprocess, time, urllib.request

MODEL_DIR   = os.path.expanduser("~/storage/models/UI-TARS-7B-DPO")
VLLM_BIN    = os.path.expanduser("~/storage/venv/bin/python")
LOG_FILE    = os.path.expanduser("~/storage/vllm.log")
PORT        = 8000

# Kill any previous vLLM process on this port
subprocess.run(f"fuser -k {PORT}/tcp", shell=True, capture_output=True)
time.sleep(2)

cmd = [
    VLLM_BIN, "-m", "vllm.entrypoints.openai.api_server",
    "--model",                MODEL_DIR,
    "--served-model-name",   "ui-tars",
    "--host",                "0.0.0.0",
    "--port",                str(PORT),
    "--max-model-len",       "32768",
    "--gpu-memory-utilization", "0.90",
    "--disable-log-requests",
]

print(f"🚀 Starting vLLM server on port {PORT} ...")
print(f"   Log: {LOG_FILE}\n")

with open(LOG_FILE, "w") as log:
    proc = subprocess.Popen(cmd, stdout=log, stderr=log)

# Poll until healthy
health_url = f"http://localhost:{PORT}/health"
print("⏳ Waiting for server to be ready", end="", flush=True)

for i in range(120):          # up to ~4 min
    time.sleep(3)
    try:
        urllib.request.urlopen(health_url, timeout=2)
        print(f"\n\n✅ vLLM is live! (took ~{(i+1)*3}s)")
        break
    except Exception:
        print(".", end="", flush=True)
else:
    print("\n\n❌ Server did not start in time. Check log:")
    print(f"   tail -50 {LOG_FILE}")

---
## Step 4 — Get Your Public URL

Lightning.ai exposes ports natively — no Cloudflare, no ngrok needed.

1. Look at the **right sidebar** in your Studio
2. Click the **Port Viewer** plugin (or **API Builder** plugin)
3. Find port **`8000`** in the list
4. Click **Public Link** → copy the URL

The URL will look like: `https://8000-<studio-id>.litng.ai`

Run the cell below to print the exact format and connection instructions.

In [ ]:
import os, socket

PORT = 8000

# Lightning.ai injects LIGHTNING_STUDIO_ID env var
studio_id = os.environ.get("LIGHTNING_STUDIO_ID", "<studio-id>")
team      = os.environ.get("LIGHTNING_TEAMSPACE_NAME", "<teamspace>")

# Construct the public URL pattern Lightning uses
public_url = f"https://{PORT}-{studio_id}.litng.ai"

print("="*60)
print("  UI-TARS SERVER IS READY")
print("="*60)
print()
print(f"  Model       : UI-TARS-7B-DPO")
print(f"  Local URL   : http://localhost:{PORT}/v1")
print(f"  Public URL  : {public_url}/v1")
print()
print("  ⚠️  Confirm the exact URL from the Port Viewer plugin")
print("      (right sidebar → Port Viewer → port 8000 → Public Link)")
print()
print("-"*60)
print("  CONNECT FROM YOUR MAC")
print("-"*60)
print()
print("  # Claude Code")
print(f"  export ANTHROPIC_BASE_URL={public_url}/v1")
print("  export ANTHROPIC_API_KEY=dummy")
print("  claude")
print()
print("  # UI-TARS Desktop")
print(f"  VLM Base URL : {public_url}/v1")
print("  Model Name   : ui-tars")
print("="*60)

---
## Step 5 — Smoke Test
_Sends a test screenshot request to confirm the model responds correctly._

In [ ]:
import urllib.request, json

PORT = 8000
URL  = f"http://localhost:{PORT}/v1/chat/completions"

payload = {
    "model": "ui-tars",
    "messages": [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "You are a GUI agent. Describe what action you would take to open a browser on a desktop."
                }
            ]
        }
    ],
    "max_tokens": 256,
    "temperature": 0.0
}

req = urllib.request.Request(
    URL,
    data=json.dumps(payload).encode(),
    headers={"Content-Type": "application/json"},
    method="POST"
)

try:
    with urllib.request.urlopen(req, timeout=60) as resp:
        data   = json.loads(resp.read())
        reply  = data["choices"][0]["message"]["content"]
        tokens = data["usage"]
        print("✅ Model responded successfully\n")
        print(f"Response:\n{reply}\n")
        print(f"Tokens — prompt: {tokens['prompt_tokens']}, completion: {tokens['completion_tokens']}")
except Exception as e:
    print(f"❌ Test failed: {e}")
    print("   Make sure Step 3 completed successfully.")

---
## Step 6 — Monitor & Manage

In [ ]:
# Tail the last 30 lines of the vLLM log
import subprocess
LOG_FILE = os.path.expanduser("~/storage/vllm.log")
result = subprocess.run(["tail", "-30", LOG_FILE], capture_output=True, text=True)
print(result.stdout)

In [ ]:
# GPU utilization check
import subprocess
result = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.used,memory.total,utilization.gpu",
     "--format=csv,noheader,nounits"],
    capture_output=True, text=True
)
lines = result.stdout.strip().split("\n")
print(f"{'GPU':<30} {'VRAM Used':>12} {'VRAM Total':>12} {'Utilization':>14}")
print("-" * 72)
for line in lines:
    parts = [p.strip() for p in line.split(",")]
    if len(parts) == 4:
        name, used, total, util = parts
        print(f"{name:<30} {used+' MB':>12} {total+' MB':>12} {util+'%':>14}")

In [ ]:
# ⚠️  Stop the server (run only when you want to shut down)
import subprocess
result = subprocess.run("fuser -k 8000/tcp", shell=True, capture_output=True, text=True)
print("🛑 vLLM server stopped." if result.returncode == 0 else "No server running on port 8000.")

---
## Notes

### After Studio restart
Steps 1 & 2 auto-skip (deps and model persist in `~/storage`).  
Just re-run **Step 3** and **Step 4** to bring the server back up.

### VRAM guide
| Model | VRAM needed | Recommended GPU |
|-------|------------|----------------|
| UI-TARS-7B-DPO | ~16 GB | L4 (24 GB) ✅ |
| UI-TARS-7B-SFT | ~16 GB | L4 (24 GB) ✅ |
| UI-TARS-72B-DPO | ~140 GB | 2× A100 80 GB |

### Changing models
Edit `MODEL_ID` in Step 2 and `MODEL_DIR` / `--served-model-name` in Step 3.

### HuggingFace gated models
If you hit a 401, set your token first:
```bash
export HF_TOKEN=hf_xxxx
```
UI-TARS-7B-DPO is public — no token needed.